In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Get the current notebook directory
current_dir = Path.cwd()

# Define the project directory
# This assumes the notebook is saved inside the "notebooks" folder
project_dir = current_dir.parent

# Define the raw data directory
raw_dir = project_dir / "data" / "raw"

print("Current notebook directory:")
print(current_dir)

print("\nProject directory:")
print(project_dir)

print("\nRaw data directory:")
print(raw_dir)

print("\nFiles in raw data folder:")
for file in raw_dir.iterdir():
    print("-", file.name)

Current notebook directory:
/Users/mac/Desktop/portfolio1_demand_inventory/notebooks

Project directory:
/Users/mac/Desktop/portfolio1_demand_inventory

Raw data directory:
/Users/mac/Desktop/portfolio1_demand_inventory/data/raw

Files in raw data folder:
- sales_train_evaluation.csv
- calendar.csv
- sell_prices.csv
- sales_train_validation.csv
- sample_submission.csv


In [2]:
# Load the three core M5 data files

calendar_path = raw_dir / "calendar.csv"
sales_path = raw_dir / "sales_train_evaluation.csv"
prices_path = raw_dir / "sell_prices.csv"

calendar = pd.read_csv(calendar_path)
sales = pd.read_csv(sales_path)
prices = pd.read_csv(prices_path)

print("Data files loaded successfully.")

print("\nCalendar data shape:")
print(calendar.shape)

print("\nSales data shape:")
print(sales.shape)

print("\nPrices data shape:")
print(prices.shape)

print("\nCalendar columns:")
print(calendar.columns.tolist())

print("\nSales columns - first 15 columns:")
print(sales.columns[:15].tolist())

print("\nSales columns - last 10 columns:")
print(sales.columns[-10:].tolist())

print("\nPrices columns:")
print(prices.columns.tolist())

print("\nCalendar first 5 rows:")
display(calendar.head())

print("\nSales first 5 rows:")
display(sales.head())

print("\nPrices first 5 rows:")
display(prices.head())

Data files loaded successfully.

Calendar data shape:
(1969, 14)

Sales data shape:
(30490, 1947)

Prices data shape:
(6841121, 4)

Calendar columns:
['date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']

Sales columns - first 15 columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd_1', 'd_2', 'd_3', 'd_4', 'd_5', 'd_6', 'd_7', 'd_8', 'd_9']

Sales columns - last 10 columns:
['d_1932', 'd_1933', 'd_1934', 'd_1935', 'd_1936', 'd_1937', 'd_1938', 'd_1939', 'd_1940', 'd_1941']

Prices columns:
['store_id', 'item_id', 'wm_yr_wk', 'sell_price']

Calendar first 5 rows:


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1



Sales first 5 rows:


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,4,0,0,0,0,3,3,0,1
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,1,2,1,1,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,2,0,0,0,2,3,0,1
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,1,0,4,0,1,3,0,2,6
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,2,1,0,0,2,1,0



Prices first 5 rows:


,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


In [3]:
# Identify high-demand products and item-store combinations

# Get all daily sales columns
day_columns = [col for col in sales.columns if col.startswith("d_")]

# ============================================================
# Part 1: Find top item-store combinations by total demand
# ============================================================

item_store_summary = sales[
    ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
].copy()

item_store_summary["total_demand"] = sales[day_columns].sum(axis=1)
item_store_summary["average_daily_demand"] = sales[day_columns].mean(axis=1)
item_store_summary["nonzero_sales_days"] = (sales[day_columns] > 0).sum(axis=1)

top_item_store = item_store_summary.sort_values(
    "total_demand", ascending=False
).head(20)

print("Top 20 item-store combinations by total demand:")
display(top_item_store)


# ============================================================
# Part 2: Find top products by total demand across all stores
# ============================================================

product_summary = item_store_summary.groupby(
    ["item_id", "dept_id", "cat_id"], as_index=False
).agg(
    total_demand=("total_demand", "sum"),
    average_daily_demand=("average_daily_demand", "mean"),
    total_nonzero_sales_days=("nonzero_sales_days", "sum"),
    number_of_stores=("store_id", "nunique")
)

top_products = product_summary.sort_values(
    "total_demand", ascending=False
).head(20)

print("\nTop 20 products by total demand across all stores:")
display(top_products)

Top 20 item-store combinations by total demand:


,id,item_id,dept_id,cat_id,store_id,state_id,total_demand,average_daily_demand,nonzero_sales_days
8412,FOODS_3_090_CA_3_evaluation,FOODS_3_090,FOODS_3,FOODS,CA_3,CA,253859,130.787738,1582
18055,FOODS_3_586_TX_2_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_2,TX,195120,100.525502,1933
21104,FOODS_3_586_TX_3_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_3,TX,151862,78.239052,1934
8908,FOODS_3_586_CA_3_evaluation,FOODS_3_586,FOODS_3,FOODS,CA_3,CA,136269,70.205564,1936
2314,FOODS_3_090_CA_1_evaluation,FOODS_3_090,FOODS_3,FOODS,CA_1,CA,128855,66.385884,1472
29755,FOODS_3_090_WI_3_evaluation,FOODS_3_090,FOODS_3,FOODS,WI_3,WI,123500,63.626996,1454
17559,FOODS_3_090_TX_2_evaluation,FOODS_3_090,FOODS_3,FOODS,TX_2,TX,121275,62.480680,1470
20608,FOODS_3_090_TX_3_evaluation,FOODS_3_090,FOODS_3,FOODS,TX_3,TX,116773,60.161257,1469
17721,FOODS_3_252_TX_2_evaluation,FOODS_3_252,FOODS_3,FOODS,TX_2,TX,115613,59.563627,1937
15006,FOODS_3_586_TX_1_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_1,TX,114010,58.737764,1927



Top 20 products by total demand across all stores:


,item_id,dept_id,cat_id,total_demand,average_daily_demand,total_nonzero_sales_days,number_of_stores
702,FOODS_3_090,FOODS_3,FOODS,1017916,52.442865,14657,10
1198,FOODS_3_586,FOODS_3,FOODS,932236,48.028645,19322,10
864,FOODS_3_252,FOODS_3,FOODS,573723,29.558114,19165,10
1167,FOODS_3_555,FOODS_3,FOODS,497881,25.650747,19250,10
1199,FOODS_3_587,FOODS_3,FOODS,402159,20.719165,15474,10
1325,FOODS_3_714,FOODS_3,FOODS,402111,20.716692,19242,10
1306,FOODS_3_694,FOODS_3,FOODS,395937,20.398609,19281,10
838,FOODS_3_226,FOODS_3,FOODS,368369,18.978310,19106,10
814,FOODS_3_202,FOODS_3,FOODS,300529,15.483205,17257,10
732,FOODS_3_120,FOODS_3,FOODS,290132,14.947553,10188,10


In [4]:
display(top_item_store)
display(top_products)

,id,item_id,dept_id,cat_id,store_id,state_id,total_demand,average_daily_demand,nonzero_sales_days
8412,FOODS_3_090_CA_3_evaluation,FOODS_3_090,FOODS_3,FOODS,CA_3,CA,253859,130.787738,1582
18055,FOODS_3_586_TX_2_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_2,TX,195120,100.525502,1933
21104,FOODS_3_586_TX_3_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_3,TX,151862,78.239052,1934
8908,FOODS_3_586_CA_3_evaluation,FOODS_3_586,FOODS_3,FOODS,CA_3,CA,136269,70.205564,1936
2314,FOODS_3_090_CA_1_evaluation,FOODS_3_090,FOODS_3,FOODS,CA_1,CA,128855,66.385884,1472
29755,FOODS_3_090_WI_3_evaluation,FOODS_3_090,FOODS_3,FOODS,WI_3,WI,123500,63.626996,1454
17559,FOODS_3_090_TX_2_evaluation,FOODS_3_090,FOODS_3,FOODS,TX_2,TX,121275,62.480680,1470
20608,FOODS_3_090_TX_3_evaluation,FOODS_3_090,FOODS_3,FOODS,TX_3,TX,116773,60.161257,1469
17721,FOODS_3_252_TX_2_evaluation,FOODS_3_252,FOODS_3,FOODS,TX_2,TX,115613,59.563627,1937
15006,FOODS_3_586_TX_1_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_1,TX,114010,58.737764,1927


,item_id,dept_id,cat_id,total_demand,average_daily_demand,total_nonzero_sales_days,number_of_stores
702,FOODS_3_090,FOODS_3,FOODS,1017916,52.442865,14657,10
1198,FOODS_3_586,FOODS_3,FOODS,932236,48.028645,19322,10
864,FOODS_3_252,FOODS_3,FOODS,573723,29.558114,19165,10
1167,FOODS_3_555,FOODS_3,FOODS,497881,25.650747,19250,10
1199,FOODS_3_587,FOODS_3,FOODS,402159,20.719165,15474,10
1325,FOODS_3_714,FOODS_3,FOODS,402111,20.716692,19242,10
1306,FOODS_3_694,FOODS_3,FOODS,395937,20.398609,19281,10
838,FOODS_3_226,FOODS_3,FOODS,368369,18.978310,19106,10
814,FOODS_3_202,FOODS_3,FOODS,300529,15.483205,17257,10
732,FOODS_3_120,FOODS_3,FOODS,290132,14.947553,10188,10


In [5]:
# Compact missing value check for the three core datasets

def missing_value_report(df, dataset_name):
    missing_counts = df.isna().sum()
    missing_columns = missing_counts[missing_counts > 0].sort_values(ascending=False)
    
    print("=" * 80)
    print(f"Dataset: {dataset_name}")
    print("=" * 80)
    print(f"Shape: {df.shape}")
    print(f"Total missing values: {missing_counts.sum()}")
    print(f"Number of columns with missing values: {len(missing_columns)}")
    
    if len(missing_columns) > 0:
        print("\nColumns with missing values:")
        print(missing_columns)
    else:
        print("\nNo missing values found.")

missing_value_report(calendar, "calendar")
missing_value_report(sales, "sales")
missing_value_report(prices, "prices")

Dataset: calendar
Shape: (1969, 14)
Total missing values: 7542
Number of columns with missing values: 4

Columns with missing values:
event_type_2    1964
event_name_2    1964
event_type_1    1807
event_name_1    1807
dtype: int64
Dataset: sales
Shape: (30490, 1947)
Total missing values: 0
Number of columns with missing values: 0

No missing values found.
Dataset: prices
Shape: (6841121, 4)
Total missing values: 0
Number of columns with missing values: 0

No missing values found.


In [6]:
# Build a daily demand table for the selected item-store combination

selected_item_id = "FOODS_3_586"
selected_store_id = "TX_2"

# Get all daily sales columns
day_columns = [col for col in sales.columns if col.startswith("d_")]

# Filter the selected item-store row from the sales dataset
selected_sales = sales[
    (sales["item_id"] == selected_item_id) &
    (sales["store_id"] == selected_store_id)
].copy()

print("Selected item-store row count:")
print(selected_sales.shape[0])

print("\nSelected item-store information:")
display(selected_sales[["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]])

# Reshape daily demand from wide format to long format
daily_demand = selected_sales[
    ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"] + day_columns
].melt(
    id_vars=["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"],
    value_vars=day_columns,
    var_name="d",
    value_name="demand"
)

# Merge with calendar data to add actual dates and time-related variables
daily_demand = daily_demand.merge(
    calendar[["d", "date", "wm_yr_wk", "weekday", "wday", "month", "year"]],
    on="d",
    how="left"
)

# Convert date from string to datetime
daily_demand["date"] = pd.to_datetime(daily_demand["date"])

# Merge with price data
daily_demand = daily_demand.merge(
    prices[["store_id", "item_id", "wm_yr_wk", "sell_price"]],
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

# Calculate daily sales revenue
daily_demand["sales_revenue"] = daily_demand["demand"] * daily_demand["sell_price"]

# Sort by date
daily_demand = daily_demand.sort_values("date").reset_index(drop=True)

print("\nDaily demand data created successfully.")

print("\nDaily demand shape:")
print(daily_demand.shape)

print("\nDate range:")
print(daily_demand["date"].min(), "to", daily_demand["date"].max())

print("\nMissing values:")
print(daily_demand.isna().sum())


print("\nFirst 10 rows:")
display(daily_demand.head(10))

print("\nLast 10 rows:")
display(daily_demand.tail(10))

Selected item-store row count:
1

Selected item-store information:


,id,item_id,dept_id,cat_id,store_id,state_id
18055,FOODS_3_586_TX_2_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_2,TX



Daily demand data created successfully.

Daily demand shape:
(1941, 16)

Date range:
2011-01-29 00:00:00 to 2016-05-22 00:00:00

Missing values:
id               0
item_id          0
dept_id          0
cat_id           0
store_id         0
state_id         0
d                0
demand           0
date             0
wm_yr_wk         0
weekday          0
wday             0
month            0
year             0
sell_price       0
sales_revenue    0
dtype: int64

First 10 rows:


,id,item_id,dept_id,cat_id,store_id,state_id,d,demand,date,wm_yr_wk,weekday,wday,month,year,sell_price,sales_revenue
0,FOODS_3_586_TX_2_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_2,TX,d_1,98,2011-01-29,11101,Saturday,1,1,2011,1.48,145.04
1,FOODS_3_586_TX_2_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_2,TX,d_2,85,2011-01-30,11101,Sunday,2,1,2011,1.48,125.80
2,FOODS_3_586_TX_2_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_2,TX,d_3,47,2011-01-31,11101,Monday,3,1,2011,1.48,69.56
3,FOODS_3_586_TX_2_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_2,TX,d_4,56,2011-02-01,11101,Tuesday,4,2,2011,1.48,82.88
4,FOODS_3_586_TX_2_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_2,TX,d_5,51,2011-02-02,11101,Wednesday,5,2,2011,1.48,75.48
5,FOODS_3_586_TX_2_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_2,TX,d_6,60,2011-02-03,11101,Thursday,6,2,2011,1.48,88.80
6,FOODS_3_586_TX_2_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_2,TX,d_7,63,2011-02-04,11101,Friday,7,2,2011,1.48,93.24
7,FOODS_3_586_TX_2_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_2,TX,d_8,96,2011-02-05,11102,Saturday,1,2,2011,1.48,142.08
8,FOODS_3_586_TX_2_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_2,TX,d_9,95,2011-02-06,11102,Sunday,2,2,2011,1.48,140.60
9,FOODS_3_586_TX_2_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_2,TX,d_10,93,2011-02-07,11102,Monday,3,2,2011,1.48,137.64



Last 10 rows:


,id,item_id,dept_id,cat_id,store_id,state_id,d,demand,date,wm_yr_wk,weekday,wday,month,year,sell_price,sales_revenue
1931,FOODS_3_586_TX_2_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_2,TX,d_1932,85,2016-05-13,11615,Friday,7,5,2016,1.68,142.80
1932,FOODS_3_586_TX_2_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_2,TX,d_1933,83,2016-05-14,11616,Saturday,1,5,2016,1.68,139.44
1933,FOODS_3_586_TX_2_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_2,TX,d_1934,103,2016-05-15,11616,Sunday,2,5,2016,1.68,173.04
1934,FOODS_3_586_TX_2_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_2,TX,d_1935,73,2016-05-16,11616,Monday,3,5,2016,1.68,122.64
1935,FOODS_3_586_TX_2_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_2,TX,d_1936,86,2016-05-17,11616,Tuesday,4,5,2016,1.68,144.48
1936,FOODS_3_586_TX_2_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_2,TX,d_1937,76,2016-05-18,11616,Wednesday,5,5,2016,1.68,127.68
1937,FOODS_3_586_TX_2_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_2,TX,d_1938,53,2016-05-19,11616,Thursday,6,5,2016,1.68,89.04
1938,FOODS_3_586_TX_2_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_2,TX,d_1939,96,2016-05-20,11616,Friday,7,5,2016,1.68,161.28
1939,FOODS_3_586_TX_2_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_2,TX,d_1940,79,2016-05-21,11617,Saturday,1,5,2016,1.68,132.72
1940,FOODS_3_586_TX_2_evaluation,FOODS_3_586,FOODS_3,FOODS,TX_2,TX,d_1941,117,2016-05-22,11617,Sunday,2,5,2016,1.68,196.56


In [7]:
# Summary statistics for demand and selling price

print("Summary statistics for demand and selling price:")
daily_demand[["demand", "sell_price"]].describe()

Summary statistics for demand and selling price:


,demand,sell_price
count,1941.000000,1941.000000
mean,100.525502,1.594529
std,30.824629,0.072191
min,0.000000,1.480000
25%,79.000000,1.580000
50%,97.000000,1.580000
75%,118.000000,1.680000
max,237.000000,1.680000


In [8]:
# Validate the daily demand time series and save the processed dataset

# Define the processed data directory
processed_dir = project_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# Check for duplicate dates
duplicate_dates = daily_demand["date"].duplicated().sum()

# Check whether dates are continuous
date_gaps = daily_demand["date"].diff().dt.days
date_gap_counts = date_gaps.value_counts(dropna=False).sort_index()

# Check zero-demand days
zero_demand_days = (daily_demand["demand"] == 0).sum()
zero_demand_rate = zero_demand_days / len(daily_demand)

# Save the processed daily demand dataset
processed_file_path = processed_dir / "daily_demand_FOODS_3_586_TX_2.csv"
daily_demand.to_csv(processed_file_path, index=False)

print("Daily demand validation completed.")

print("\nNumber of rows:")
print(len(daily_demand))

print("\nNumber of duplicate dates:")
print(duplicate_dates)

print("\nDate gap distribution:")
print(date_gap_counts)

print("\nZero-demand days:")
print(zero_demand_days)

print("\nZero-demand rate:")
print(round(zero_demand_rate, 4))

print("\nProcessed file saved to:")
print(processed_file_path)

Daily demand validation completed.

Number of rows:
1941

Number of duplicate dates:
0

Date gap distribution:
date
1.0    1940
NaN       1
Name: count, dtype: int64

Zero-demand days:
8

Zero-demand rate:
0.0041

Processed file saved to:
/Users/mac/Desktop/portfolio1_demand_inventory/data/processed/daily_demand_FOODS_3_586_TX_2.csv
